In [126]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision.utils import make_grid
import torchvision 
from torchvision.datasets import ImageFolder
import torchvision.transforms as transforms
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime
import os
from PIL import Image
import cv2

In [127]:
# finding the dimensions to resize images to

def zerosForName(s):
    if(len(s) == 4):
        return s
    elif(len(s) == 3):
        return "0" + s
    elif(len(s) == 2):
        return "00" + s
    elif(len(s) == 1):
        return "000" + s

data_dir = '../input/brain-tumor-mri-dataset/Training'
classes = os.listdir(data_dir)

minminheight, minminwidth = 1000, 1000

for test_train in ["Testing", "Training"]:
    
    for brain in classes:

        with Image.open("../input/brain-tumor-mri-dataset/"+test_train+"/"+brain+"/"+test_train[0:2]+"-"+brain[0:2]+"Tr_0000.jpg") as img:
            minwidth, minheight = img.size

        for i in range(1, 10):
            with Image.open("../input/brain-tumor-mri-dataset/"+test_train+"/"+brain+"/"+test_train[0:2]+"-"+brain[0:2]+"Tr_"+zerosForName(str(i))+".jpg") as img:
                width, height = img.size
                if(height < minheight):
                    minheight = height
                if(width < minwidth):
                    minwidth = width

        for i in range(10, 2000):
            try:
                with Image.open("../input/brain-tumor-mri-dataset/"+test_train+"/"+brain+"/"+test_train[0:2]+"-"+brain[0:2]+"_"+zerosForName(str(i))+".jpg") as img:
                    width, height = img.size
                    if(height < minheight):
                        minheight = height
                    if(width < minwidth):
                        minwidth = width
            except:
                pass

        #print("Min height and widths:", minheight, minwidth, brain, test_train)   
        
        if(minheight < minminheight):
            minminheight = minheight
        if(minwidth < minminwidth):
            minminwidth = minwidth

print("min h:", minminheight, "min w:", minminwidth)

In [128]:
K = len(classes)
print("number of classes:", K)

In [129]:
model = nn.Sequential(
    nn.Conv2d(in_channels=3, out_channels=32, kernel_size=3, stride=1),
    nn.ReLU(),
    nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3, stride=1),
    nn.ReLU(),
    nn.Conv2d(in_channels=64, out_channels=128, kernel_size=3, stride=1),
    nn.Flatten(),
    nn.Dropout(p=0.2),
    nn.Linear(128*22*22, 128),
    nn.ReLU(),
    nn.Linear(128, 64),
    nn.ReLU(),
    nn.Dropout(p=0.2),
    nn.Linear(64, K)
)

In [130]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(device)
model.to(device)

In [131]:
train_transform=transforms.Compose([
#         transforms.RandomRotation(10),      # rotate +/- 10 degrees
#         transforms.RandomHorizontalFlip(),  # reverse 50% of images
        transforms.Resize((28,28)),             # resize shortest side
        #transforms.CenterCrop(80),         # crop longest side
        transforms.ToTensor(),
#         transforms.Normalize([0.485, 0.456, 0.406],
#                              [0.229, 0.224, 0.225])
])

In [132]:
dataset = ImageFolder("../input/brain-tumor-mri-dataset/Training", transform=train_transform)
testset = ImageFolder("../input/brain-tumor-mri-dataset/Testing", transform=train_transform)

In [133]:
batch_size = 16

train_loader = DataLoader(
    dataset=dataset, 
    batch_size=batch_size, 
    shuffle=True
)

test_loader = DataLoader(
    dataset=testset, 
    batch_size=batch_size, 
    shuffle=False
)

In [134]:
for images, labels in train_loader:
    fig, ax = plt.subplots(figsize=(18,10))
    ax.set_xticks([])
    ax.set_yticks([])
    ax.imshow(make_grid(images, nrow=16).permute(1, 2, 0))
    break

In [135]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters())

In [136]:
def batch_gd(model, criterion, optimizer, train_loader, test_loader, epochs):
    train_losses = np.zeros(epochs)
    test_losses = np.zeros(epochs)

    for it in range(epochs):
        model.train()
        t0 = datetime.now()
        train_loss = []
        for inputs, targets in train_loader:
        # move data to GPU
            inputs, targets = inputs.to(device), targets.to(device)

            # zero the parameter gradients
            optimizer.zero_grad()

            # Forward pass
            outputs = model(inputs)
            loss = criterion(outputs, targets)

            # Backward and optimize
            loss.backward()
            optimizer.step()

            train_loss.append(loss.item())

            # Get train loss and test loss
        train_loss = np.mean(train_loss) # a little misleading

        model.eval()
        test_loss = []
        for inputs, targets in test_loader:
            inputs, targets = inputs.to(device), targets.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, targets)
            test_loss.append(loss.item())
        test_loss = np.mean(test_loss)

        # Save losses
        train_losses[it] = train_loss
        test_losses[it] = test_loss

        dt = datetime.now() - t0
        print(f'Epoch {it+1}/{epochs}, Train Loss: {train_loss:.4f}, \Test Loss: {test_loss:.4f}, Duration: {dt}')

    return train_losses, test_losses

train_losses, test_losses = batch_gd(model, criterion, optimizer, train_loader, test_loader, epochs=5)

In [137]:
plt.plot(train_losses, label='train loss')
plt.plot(test_losses, label='test loss')
plt.legend()
plt.show()

In [138]:
model.eval()
n_correct = 0.
n_total = 0.
for inputs, targets in train_loader:
  # move data to GPU
  inputs, targets = inputs.to(device), targets.to(device)

  # Forward pass
  outputs = model(inputs)

  # Get prediction
  # torch.max returns both max and argmax
  _, predictions = torch.max(outputs, 1)
  
  # update counts
  n_correct += (predictions == targets).sum().item()
  n_total += targets.shape[0]

train_acc = n_correct / n_total


n_correct = 0.
n_total = 0.
for inputs, targets in test_loader:
  # move data to GPU
  inputs, targets = inputs.to(device), targets.to(device)

  # Forward pass
  outputs = model(inputs)

  # Get prediction
  # torch.max returns both max and argmax
  _, predictions = torch.max(outputs, 1)
  
  # update counts
  n_correct += (predictions == targets).sum().item()
  n_total += targets.shape[0]

test_acc = n_correct / n_total
print(f"Train acc: {train_acc:.4f}, Test acc: {test_acc:.4f}")